## 1. Setup

In [8]:
from pathlib import Path
import pandas as pd
import numpy as np
import yaml
import torch
import cv2
from ultralytics import YOLO

RAW_DIR = Path("/kaggle/input/datasets/stevenhicks/visem-video-dataset/visem-dataset")
VIDEOS_DIR = RAW_DIR / "videos"
VISION_DIR = Path("data/vision")
MODEL_PATH = Path("/kaggle/input/models/tazana/dlfinal-yolo/pytorch/default/1/best-final.pt")

# Si todavía tienes el modelo dentro de notebooks/
if not MODEL_PATH.exists():
    MODEL_PATH = Path("../notebooks/best-final.pt")

VISION_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("VIDEOS_DIR:", VIDEOS_DIR)
print("VISION_DIR:", VISION_DIR)
print("MODEL_PATH:", MODEL_PATH)
print("DEVICE:", DEVICE)

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"No se encontró el modelo: {MODEL_PATH}")

if not VIDEOS_DIR.exists():
    raise FileNotFoundError(
        f"No existe {VIDEOS_DIR}. Debes poner ahí los .avi del dataset."
    )

VIDEOS_DIR: /kaggle/input/datasets/stevenhicks/visem-video-dataset/visem-dataset/videos
VISION_DIR: data/vision
MODEL_PATH: /kaggle/input/models/tazana/dlfinal-yolo/pytorch/default/1/best-final.pt
DEVICE: 0


## 2. Leer videos.csv

In [9]:
videos_df = pd.read_csv(
    RAW_DIR / "videos.csv",
    sep=";",
    decimal=",",
    encoding="utf-8"
)

videos_df.columns = [c.strip().lower() for c in videos_df.columns]
videos_df = videos_df.rename(columns={"id": "id", "video": "video_filename"})

videos_df["id"] = videos_df["id"].astype(int)
videos_df["video_filename"] = videos_df["video_filename"].astype(str)

videos_df.head()

,id,video_filename
0,1,1_09.09.02_SSW.avi
1,2,2_09.09.03_lots of debris_SSW.avi
2,3,3_11.01.21_JMA.avi
3,4,4_11.03.29_HH.avi
4,5,5_11.05.04_JMA.avi


## 3. Verificar que existan los videos

In [10]:
videos_df["video_path"] = videos_df["video_filename"].apply(
    lambda name: VIDEOS_DIR / name
)

videos_df["video_exists"] = videos_df["video_path"].apply(lambda p: p.exists())

print(videos_df["video_exists"].value_counts())

display(videos_df[videos_df["video_exists"] == False].head(20))

video_exists
True    85
Name: count, dtype: int64


,id,video_filename,video_path,video_exists


## 4. Crear configuración de ByteTrack
Este archivo define cómo ByteTrack va a unir detecciones entre frames. Ultralytics usa archivos YAML para configurar trackers como ByteTrack o BoT-SORT.

In [14]:
tracker_yaml_path = VISION_DIR / "bytetrack_visem.yaml"

tracker_config = {
    "tracker_type": "bytetrack",
    "track_high_thresh": 0.30,
    "track_low_thresh": 0.10,
    "new_track_thresh": 0.30,
    "track_buffer": 60,
    "match_thresh": 0.80,
    "fuse_score": True,
}

with open(tracker_yaml_path, "w") as f:
    yaml.safe_dump(tracker_config, f, sort_keys=False)

print("Tracker config guardada en:", tracker_yaml_path)

Tracker config guardada en: data/vision/bytetrack_visem.yaml


## 5. Correr YOLO + ByteTrack sobre todos los videos

In [31]:
model = YOLO(str(MODEL_PATH))

MAX_SECONDS = 120 # Procesar solo los primeros 120 segundos de cada video para acelerar el proceso
TARGET_CLASS_ID = 0
IMG_SIZE = 640 # Antes estaba en 960 pero demoraba demasiado en procesar
CONF = 0.25
IOU = 0.50

# Carpeta donde se guardará UN CSV POR VIDEO
TRACKS_DIR = VISION_DIR / "tracks_per_video"
TRACKS_DIR.mkdir(exist_ok=True)

# Filtrar videos válidos
valid_videos_df = videos_df[
    videos_df["video_exists"]
].copy()

print("Videos válidos:", len(valid_videos_df))


# Procesar videos
for _, row in valid_videos_df.iterrows():

    participant_id = int(row["id"])
    video_filename = row["video_filename"]
    video_path = row["video_path"]

    # Nombre seguro para guardar csv
    safe_name = Path(video_filename).stem

    output_csv = TRACKS_DIR / f"{safe_name}.csv"

    # Saltar videos ya procesados
    if output_csv.exists():
        print(f"SKIP -> {video_filename}")
        continue

    print(f"Procesando: {participant_id} - {video_filename}".center(100, "="))

    all_rows_video = []

    try:
        cap = cv2.VideoCapture(str(video_path))
        fps = cap.get(cv2.CAP_PROP_FPS)
        cap.release()
        if fps <= 0:
            fps = 30  # fallback por si el AVI no reporta bien el FPS

        max_frames = int(MAX_SECONDS * fps)
        print("FPS:", fps)
        print("Procesando máximo frames:", max_frames)
        results_stream = model.track(
            source=str(video_path),
            stream=True,
            tracker=str(tracker_yaml_path),
            imgsz=IMG_SIZE,
            conf=CONF,
            iou=IOU,
            device=DEVICE,
            persist=False,
            verbose=False,
            save=False,
            classes=[TARGET_CLASS_ID],
            half=True,
            max_det=60,
        )

        n_detections = 0

        for frame_idx, result in enumerate(results_stream):
            if frame_idx >= max_frames:
                print(f"Cortando en {MAX_SECONDS} segundos ({max_frames} frames)")
                break

            if result.boxes is None:
                continue

            if len(result.boxes) == 0:
                continue

            if result.boxes.id is None:
                continue

            xywhn = result.boxes.xywhn.cpu().numpy()
            confs = result.boxes.conf.cpu().numpy()
            ids = result.boxes.id.cpu().numpy().astype(int)

            for box, conf, track_id in zip(
                xywhn,
                confs,
                ids
            ):

                x_center, y_center, width, height = box

                all_rows_video.append({
                    "id": participant_id,
                    "video": str(participant_id),
                    "video_filename": video_filename,
                    "frame": int(frame_idx),
                    "track_id": str(track_id),
                    "x_center": float(x_center),
                    "y_center": float(y_center),
                    "width": float(width),
                    "height": float(height),
                    "confidence": float(conf),
                    "source_video_path": str(video_path),
                })

                n_detections += 1

        print("Detecciones con track_id:", n_detections)

        # Guardar CSV por video
        if len(all_rows_video) > 0:

            video_df = pd.DataFrame(all_rows_video)

            video_df["track_uid"] = (
                video_df["video_filename"].astype(str)
                + "__"
                + video_df["track_id"].astype(str)
            )

            video_df.to_csv(
                output_csv,
                index=False
            )

            print("Guardado:", output_csv)

        else:
            print("Sin detecciones:", video_filename)

    except Exception as e:

        print(f"ERROR en {video_filename}")
        print(e)

Videos válidos: 85
=================================Procesando: 1 - 1_09.09.02_SSW.avi=================================
FPS: 49.576
Procesando máximo frames: 5949
Cortando en 120 segundos (5949 frames)
Detecciones con track_id: 348260
Guardado: data/vision/tracks_per_video/1_09.09.02_SSW.csv
=========================Procesando: 2 - 2_09.09.03_lots of debris_SSW.avi==========================
FPS: 48.302
Procesando máximo frames: 5796
Cortando en 120 segundos (5796 frames)
Detecciones con track_id: 142391
Guardado: data/vision/tracks_per_video/2_09.09.03_lots of debris_SSW.csv
=================================Procesando: 3 - 3_11.01.21_JMA.avi=================================
FPS: 49.977
Procesando máximo frames: 5997
Cortando en 120 segundos (5997 frames)
Detecciones con track_id: 299848
Guardado: data/vision/tracks_per_video/3_11.01.21_JMA.csv
=================================Procesando: 4 - 4_11.03.29_HH.avi==================================
FPS: 49.985
Procesando máximo frames: 5998


## 6. Juntar todos los CSVs en uno solo

In [32]:
print("\nUniendo CSVs...")

all_csvs = list(TRACKS_DIR.glob("*.csv"))

if len(all_csvs) > 0:

    tracks_df = pd.concat(
        [pd.read_csv(csv_path) for csv_path in all_csvs],
        ignore_index=True
    )

    final_output = VISION_DIR / "pred_tracks_all_videos.csv"

    tracks_df.to_csv(
        final_output,
        index=False
    )

    print("\n=========================")
    print("TRACKS FINALES GUARDADOS")
    print("=========================")
    print("Archivo:", final_output)
    print("Shape:", tracks_df.shape)

    display(tracks_df.head())

else:

    print("No se encontraron CSVs.")


Uniendo CSVs...

TRACKS FINALES GUARDADOS
Archivo: data/vision/pred_tracks_all_videos.csv
Shape: (22343478, 12)


,id,video,video_filename,frame,track_id,x_center,y_center,width,height,confidence,source_video_path,track_uid
0,52,52,52_09.12.03_drift_SSW.avi,0,1,0.464844,0.463021,0.035937,0.054688,0.861328,/kaggle/input/datasets/stevenhicks/visem-video...,52_09.12.03_drift_SSW.avi__1
1,52,52,52_09.12.03_drift_SSW.avi,0,2,0.397070,0.497917,0.027734,0.036458,0.812988,/kaggle/input/datasets/stevenhicks/visem-video...,52_09.12.03_drift_SSW.avi__2
2,52,52,52_09.12.03_drift_SSW.avi,0,3,0.516406,0.647396,0.039062,0.048958,0.721680,/kaggle/input/datasets/stevenhicks/visem-video...,52_09.12.03_drift_SSW.avi__3
3,52,52,52_09.12.03_drift_SSW.avi,0,4,0.520312,0.101237,0.026563,0.040885,0.665527,/kaggle/input/datasets/stevenhicks/visem-video...,52_09.12.03_drift_SSW.avi__4
4,52,52,52_09.12.03_drift_SSW.avi,0,5,0.022900,0.622396,0.044092,0.062500,0.607422,/kaggle/input/datasets/stevenhicks/visem-video...,52_09.12.03_drift_SSW.avi__5


## 7. Validar tracks generados

In [4]:
import pandas as pd
from pathlib import Path
VISION_DIR = Path("../data/vision")

In [ ]:
tracks_df = pd.read_csv(VISION_DIR / "pred_tracks_all_videos.csv")

print("Filas:", len(tracks_df))
print("Participantes con tracks:", tracks_df["id"].nunique())
print("Videos con tracks:", tracks_df["video_filename"].nunique())
print("Tracks únicos:", tracks_df["track_uid"].nunique())

display(
    tracks_df
    .groupby(["id", "video_filename"])
    .agg(
        n_points=("frame", "count"),
        n_frames=("frame", "nunique"),
        n_tracks=("track_uid", "nunique"),
        mean_confidence=("confidence", "mean"),
    )
    .reset_index()
    .sort_values("n_tracks", ascending=False)
    .head(20)
)

Filas: 22343478
Participantes con tracks: 85
Videos con tracks: 85
Tracks únicos: 703106


,id,video_filename,n_points,n_frames,n_tracks,mean_confidence
30,31,31_10.04.22_JMA.avi,255602,5710,29834,0.579155
26,27,27_09.02.18_IVS.avi,285393,5969,27840,0.715156
15,16,16_09.01.28_SSW.avi,285697,5916,27641,0.686606
82,83,83_11.11.09_HH.avi,268742,5754,27514,0.655550
25,26,26_09.02.12_IVS.avi,295323,5942,27056,0.684169
...,...,...,...,...,...,...
51,52,52_09.12.03_drift_SSW.avi,75676,5762,230,0.657694
46,47,47_09.10.19_SSW.avi,37257,5969,214,0.602239
13,14,14_09.01.27_SSW.avi,26502,5989,111,0.616224
22,23,23_09.02.04_SSW.avi,26250,5683,91,0.620507
